<a href="https://colab.research.google.com/github/rizz01107/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rizz01107/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content page's daily search performance record — a specific (client, content page, date) combination from fact_content_daily_performance.

Time window: month=2026-03 (a mid-panel month, per the warning to never use the final month for label logic — that's the sealed test month).

In [3]:
FACT_MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

# Grain check: is one row really one (client, content, date)?
con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) as n
    FROM {FACT_MAR}
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬─────────────────┬─────────────┬───────┐
│ client_hash_id │ content_hash_id │ report_date │   n   │
│    varchar     │     varchar     │    date     │ int64 │
├────────────────┴─────────────────┴─────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature: gsc_impressions, gsc_clicks, gsc_sum_position (all pre-decision, observable signals)
Label/proxy: whether clicks declined from the first half of the month to the second half (built only from this month's own data, for teaching purposes)
Context: client_hash_id, content_hash_id, report_date, month
Excluded: ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other — excluded because AI-referral sessions are sparse per client/page and unreliable as a per-page signal at this grain (per the warehouse notes); scroll_events excluded for this lane since it's behavioral and not consistently available across all pages.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three verification queries below prove the contract's claims. Then a 5-feature frame (each computed only from days 1-15, so it's knowable before the "decision moment" of day 15). Then the trap: adding a feature from the label's own time window (days 16-31) and watching the score jump — then removing it.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
    SELECT COUNT(*) as row_count,
           MIN(report_date) as min_date,
           MAX(report_date) as max_date
    FROM {FACT_MAR}
""").show()

┌───────────┬────────────┬────────────┐
│ row_count │  min_date  │  max_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘



In [5]:
con.sql(f"""
    SELECT COUNT(*) as total_rows
    FROM {FACT_MAR}
""").show()

con.sql(f"""
    SELECT COUNT(*) as available_rows
    FROM {FACT_MAR}
    WHERE gsc_data_available IS TRUE
""").show()

┌────────────┐
│ total_rows │
│   int64    │
├────────────┤
│    9841378 │
└────────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┐
│ available_rows │
│     int64      │
├────────────────┤
│        3611061 │
└────────────────┘



In [6]:
features = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS imp_first15,
           SUM(gsc_clicks) AS clk_first15,
           AVG(gsc_sum_position / NULLIF(gsc_impressions,0)) AS avg_position_first15,
           SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions),0) AS ctr_first15,
           MAX(client_has_ga4::INT) AS has_ga4
    FROM {FACT_MAR}
    WHERE gsc_data_available IS TRUE
      AND CAST(SUBSTR(CAST(report_date AS VARCHAR), 9, 2) AS INT) <= 15
    GROUP BY 1,2
    HAVING imp_first15 >= 50
""").df()
print(f"{len(features):,} content items with enough early-month history")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

92,548 content items with enough early-month history


,client_hash_id,content_hash_id,imp_first15,clk_first15,avg_position_first15,ctr_first15,has_ga4
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,429.0,2.0,4.247255,0.004662,0
1,client_73cda7b4e4f265ea,content_905aa32a0230694e,89.0,0.0,3.010741,0.000000,0
2,client_73cda7b4e4f265ea,content_05434271b257bb68,628.0,1.0,5.330069,0.001592,0
3,client_73cda7b4e4f265ea,content_d056587ff7faca0c,1280.0,9.0,4.468441,0.007031,0
4,client_73cda7b4e4f265ea,content_2662845f598544ef,97.0,0.0,8.765983,0.000000,0


In [7]:
label_data = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_clicks) AS clk_last15
    FROM {FACT_MAR}
    WHERE gsc_data_available IS TRUE
      AND CAST(SUBSTR(CAST(report_date AS VARCHAR), 9, 2) AS INT) > 15
    GROUP BY 1,2
""").df()

data = features.merge(label_data, on=['client_hash_id','content_hash_id'], how='inner')
data['is_declining'] = (data['clk_last15'] < data['clk_first15']).astype(int)
print(f"{len(data):,} rows | declining rate: {data['is_declining'].mean():.3f}")

92,247 rows | declining rate: 0.289


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import numpy as np

honest_features = ['imp_first15','clk_first15','avg_position_first15','ctr_first15','has_ga4']
X_honest = data[honest_features].replace([np.inf,-np.inf], np.nan).fillna(0)
y = data['is_declining']

model_honest = LogisticRegression(max_iter=500).fit(X_honest, y)
auc_honest = roc_auc_score(y, model_honest.predict_proba(X_honest)[:,1])
print(f"Honest AUC (5 features only): {auc_honest:.3f}")

# THE TRAP: add clk_last15 — a column FROM the label's own time window
data['leaky_feature'] = data['clk_last15']
X_leaky = data[honest_features + ['leaky_feature']].replace([np.inf,-np.inf], np.nan).fillna(0)
model_leaky = LogisticRegression(max_iter=500).fit(X_leaky, y)
auc_leaky = roc_auc_score(y, model_leaky.predict_proba(X_leaky)[:,1])
print(f"'Leaky' AUC (with clk_last15 added): {auc_leaky:.3f}  <- jumps toward perfect")

# Delete it, keep the honest number
del data['leaky_feature']
print(f"\nKept: honest AUC = {auc_honest:.3f} (not the leaky {auc_leaky:.3f})")

Honest AUC (5 features only): 0.782
'Leaky' AUC (with clk_last15 added): 1.000  <- jumps toward perfect

Kept: honest AUC = 0.782 (not the leaky 1.000)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice can never tell you whether the decline is caused by anything actionable — only correlational. History depth is unbalanced across clients (gsc_data_start varies), so a mid-panel month may represent different amounts of each client's true history. Only 37% of rows (3,611,061 of 9,841,378) have gsc_data_available IS TRUE — the rest are gaps, not zeros. AI-referral columns (ai_chatgpt, ai_claude, etc.) were excluded entirely since they are known to be sparse per client/page. The first-half/second-half split used here is a same-month proxy for teaching leakage — it is not the real forward-looking label used in later capstone work, where the label must come from a genuinely future month.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.